# 03 - Privacy Demonstration

This notebook evaluates privacy risks in the **raw credit application dataset** and demonstrates how personally identifiable information (PII) can be protected through pseudonymization, anonymization, and governance-oriented data controls.

Unlike the curated analytical dataset used in the previous notebooks, this notebook works with the **raw dataset**, which preserves the original structure of the application records and still contains personal identifiers that must be evaluated from a privacy and governance perspective.

The goals of the notebook are:

- Identify **personally identifiable information (PII)** in the dataset.
- Demonstrate **pseudonymization** of sensitive identifiers.
- Demonstrate **data minimization / anonymization** by removing unnecessary fields.
- Map the findings to **GDPR principles** such as lawful basis, data minimization, and storage limitation.
- Illustrate governance controls relevant to **high-risk AI systems** under the EU AI Act.

This privacy audit is conducted on the raw dataset **before the creation of the curated analytical dataset used in the other notebooks**, allowing potential identifiers to be detected and mitigated before analytical use.

In [10]:
import json
import hashlib
import pandas as pd
from datetime import datetime, timedelta

## 1) Load the Raw Dataset

The raw JSON dataset is loaded and inspected.  
This dataset preserves the original application structure and therefore allows a privacy audit to detect direct and indirect identifiers.

In [12]:
with open("raw_credit_applications.json", "r") as f:
    file_data = json.load(f)

# remove duplicate IDs if any
seen_ids = set()
unique_documents = []

for doc in file_data:
    if "_id" in doc:
        if doc["_id"] not in seen_ids:
            unique_documents.append(doc)
            seen_ids.add(doc["_id"])
    else:
        unique_documents.append(doc)

print(f"Dataset loaded: {len(unique_documents)} records")

print("\nExample structure:")
print(unique_documents[0].keys())

Dataset loaded: 500 records

Example structure:
dict_keys(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp'])


## 2) Identification of Personally Identifiable Information (PII)

The raw dataset may contain several categories of personal data:

- **Direct identifiers**: name, email, social security number
- **Online identifiers**: IP address
- **Quasi-identifiers**: date of birth, ZIP code, gender
- **Financial attributes**: income and spending behavior

Even when direct identifiers are removed, combinations of quasi-identifiers may still allow individuals to be re-identified.

In [13]:
def get_nested_value(data, path):
    keys = path.split(".")
    val = data
    for k in keys:
        if isinstance(val, dict):
            val = val.get(k, None)
        elif isinstance(val, list):
            val = [item.get(k) for item in val if isinstance(item, dict)]
        else:
            return None
    return val


pii_inventory = {
    "Direct Identifiers": [
        "applicant_info.full_name",
        "applicant_info.email",
        "applicant_info.ssn",
        "app_id"
    ],
    "Online Identifiers": [
        "applicant_info.ip_address"
    ],
    "Quasi Identifiers": [
        "applicant_info.date_of_birth",
        "applicant_info.zip_code",
        "applicant_info.gender"
    ],
    "Financial Data": [
        "financials.annual_income",
        "spending_behavior.category"
    ]
}

print("PII AUDIT REPORT")
print("="*40)

for category, fields in pii_inventory.items():
    print(f"\n{category}")

    for field in fields:
        count = 0
        example = None

        for doc in unique_documents:
            val = get_nested_value(doc, field)

            if val not in [None, [], ""]:
                count += 1
                if example is None:
                    example = val

        if count > 0:
            print(f"[!] {field} exposed in {count} records")
            print(f"Example: {example}")
        else:
            print(f"[OK] {field} not found")

PII AUDIT REPORT

Direct Identifiers
[!] applicant_info.full_name exposed in 500 records
Example: Jerry Smith
[!] applicant_info.email exposed in 493 records
Example: jerry.smith17@hotmail.com
[!] applicant_info.ssn exposed in 496 records
Example: 596-64-4340
[OK] app_id not found

Online Identifiers
[!] applicant_info.ip_address exposed in 496 records
Example: 192.168.48.155

Quasi Identifiers
[!] applicant_info.date_of_birth exposed in 496 records
Example: 2001-03-09
[!] applicant_info.zip_code exposed in 499 records
Example: 10036
[!] applicant_info.gender exposed in 498 records
Example: Male

Financial Data
[!] financials.annual_income exposed in 495 records
Example: 73000
[!] spending_behavior.category exposed in 500 records
Example: ['Shopping', 'Rent', 'Alcohol']


The audit confirms that the raw dataset contains several categories of personal data, including direct identifiers such as full name, email, and Social Security Number, as well as online and quasi-identifiers such as IP address, date of birth, ZIP code, and gender.

From a privacy perspective, this means the raw dataset is not suitable for analytical use without additional safeguards. Sensitive fields should be pseudonymized or removed before the data is used for downstream analysis.

## 3) Pseudonymization of Sensitive Identifiers

A common privacy-preserving technique under the GDPR is **pseudonymization**, in which directly identifiable values are replaced by transformed values that no longer reveal the identity of the data subject in plain text.

In this example, the fields **email** and **SSN** are transformed using the SHA-256 hashing algorithm. This reduces direct identifiability while preserving a stable transformed value that could still support internal record linkage if needed.

It is important to note that pseudonymization is not the same as anonymization. A pseudonymized dataset may still constitute personal data under the GDPR if re-identification remains possible.

In [14]:
def pseudonymize(value):
    if value and isinstance(value, str):
        return hashlib.sha256(value.encode()).hexdigest()
    return value

privacy_docs = [doc.copy() for doc in unique_documents]

expiry_date = (datetime.now() + timedelta(days=5*365)).strftime("%Y-%m-%d")

for doc in privacy_docs:
    app_info = doc.get("applicant_info", {})

    if "email" in app_info:
        app_info["email"] = pseudonymize(app_info["email"])

    if "ssn" in app_info:
        app_info["ssn"] = pseudonymize(app_info["ssn"])

    doc["privacy_status"] = "pseudonymized"

    doc["retention_policy"] = {
        "expiry_date": expiry_date,
        "action": "delete_after_expiry"
    }

sample = privacy_docs[0]

print("PSEUDONYMIZATION EVIDENCE")
print("=" * 40)
print("Hashed email:", sample["applicant_info"].get("email"))
print("Hashed ssn:", sample["applicant_info"].get("ssn"))

PSEUDONYMIZATION EVIDENCE
Hashed email: 116648a7761525746032d0ab323ceb01f50d11f793516437a5cf80804fc6fb9d
Hashed ssn: 2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333ef8ddabb44b2511ccbd


The transformation above replaces direct identifiers with SHA-256 hashes, meaning that the original values are no longer visible in plain text.

This supports the GDPR principle of **security of processing** because sensitive identifiers are protected from direct exposure. At the same time, the dataset is not fully anonymous, since other attributes remain and could still contribute to re-identification risk.

## 4) Data Minimization and Anonymization

The GDPR principle of **data minimization** requires organizations to retain only data that is necessary for the intended purpose.

In the context of credit application analysis, highly specific identifiers such as IP addresses may not be necessary once fraud prevention or operational verification steps are completed. Removing such fields reduces privacy risk and improves compliance with the principle of proportional data use.

In the following step, the IP address field is removed from the dataset.

In [15]:
for doc in privacy_docs:
    app_info = doc.get("applicant_info", {})

    if "ip_address" in app_info:
        del app_info["ip_address"]

sample_after = privacy_docs[0]

print("DATA MINIMIZATION CHECK")
print("=" * 40)
print("IP address present?", "ip_address" in sample_after.get("applicant_info", {}))

DATA MINIMIZATION CHECK
IP address present? False


## 5) GDPR Mapping

The dataset and analysis can be evaluated against key principles established by the General Data Protection Regulation (GDPR).

### Lawful Basis

Credit institutions may process applicant data under **contractual necessity**, as the information is required to evaluate creditworthiness and determine loan approval decisions.

### Data Minimization (GDPR Art. 5(1)(c))

The curated dataset retains only attributes necessary for analyzing loan approval outcomes and fairness patterns. Direct identifiers are not included, which reduces privacy risk.

Additional minimization can be achieved through transformations such as grouping ages into ranges and generalizing geographic identifiers.

### Storage Limitation

Personal data should not be retained longer than necessary for the intended purpose. Analytical datasets should store only the minimum information required and avoid retaining identifiable information where possible.

## 6) Governance Implications

Even when direct identifiers are removed, datasets may still present privacy risks through indirect identifiers such as demographic attributes or geographic indicators.

Responsible data governance therefore requires mechanisms that protect individuals while allowing meaningful analysis. These mechanisms include data minimization, anonymization techniques, access controls, and transparent documentation of how data is used.

In the context of credit decision systems, privacy protection is closely linked to broader governance requirements such as fairness monitoring, transparency, and human oversight. Implementing such measures helps organizations comply with GDPR principles and aligns with governance expectations for high-risk AI systems under the EU AI Act.

## Conclusion

This privacy demonstration identified potential quasi-identifiers in the credit application dataset and illustrated how privacy-preserving transformations can reduce re-identification risk.

By applying generalization techniques to attributes such as age and ZIP code, the dataset maintains analytical usefulness while improving privacy protection.

These transformations support GDPR principles such as data minimization and storage limitation and contribute to responsible data governance in automated credit decision systems.